In [2]:
import pandas as pd
import sqlite3
from pathlib import Path

# chemin vers le fichier nettoyé
csv_path = Path("../data/cleaned_data.csv")

# charger les données
df = pd.read_csv(csv_path, parse_dates=["InvoiceDate"])

print("Dimensions :", df.shape)
print(df.head())
print(df.dtypes)

Dimensions : (536641, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice     

In [4]:
# créer ou ouvrir la base de données locale
conn = sqlite3.connect("../analysis.db")

# transférer le DataFrame vers SQLite
df.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

print("Données transférer dans SQLite avec succès !")

Données transférer dans SQLite avec succès !


In [6]:
pd.read_sql("SELECT COUNT(*) AS nombre_lignes FROM sales", conn)

,nombre_lignes
0,536641


In [11]:
# les 5 requêtes SPÉCIFIQUES
# top 10 des pays par chiffre d'affaire total: Quelles sont les dix pays qui génèrent le plus de chiffre d'affaires net ?
query1 = """
SELECT
    Country,
    ROUND(SUM(Quantity * UnitPrice), 2) AS chiffre_affaires
FROM sales
GROUP BY country
ORDER BY chiffre_affaires DESC
LIMIT 10;
"""

resultat1 = pd.read_sql(query1, conn)
resultat1
    

,Country,chiffre_affaires
0,United Kingdom,8167128.18
1,Netherlands,284661.54
2,EIRE,262993.38
3,Germany,221509.47
4,France,197317.11
5,Australia,137009.77
6,Switzerland,56363.05
7,Spain,54756.03
8,Belgium,40910.96
9,Sweden,36585.41


In [12]:
# Chiffre d'affaire par mois 
#Quelles sont les mois les plus performants et comment évoluent les ventes au cour des temps
query2 = """
SELECT
    strftime('%Y-%m', InvoiceDate) AS mois,
    ROUND(SUM(Quantity * UnitPrice), 2) AS chiffre_affaires
FROM sales
GROUP BY strftime('%Y-%m', InvoiceDate)
ORDER BY mois;
"""

resultat2 = pd.read_sql(query2, conn)
resultat2

,mois,chiffre_affaires
0,2010-12,746723.61
1,2011-01,558448.56
2,2011-02,497026.41
3,2011-03,682013.98
4,2011-04,492367.84
5,2011-05,722094.10
6,2011-06,689977.23
7,2011-07,680156.99
8,2011-08,681386.46
9,2011-09,1017596.68


In [13]:
# Heures de pointe des achats
# A quelles heures les clients passent-il le plus de commandes ?
query3 = """
SELECT
    strftime('%H', InvoiceDate) AS heure,
    COUNT(DISTINCT InvoiceNo) AS nombre_commandes
FROM sales
WHERE Quantity > 0
    AND UnitPrice > 0
    AND InvoiceNo NOT LIKE 'C%'
GROUP BY strftime('%H', InvoiceDate)
ORDER BY nombre_commandes DESC;
"""


resultat3 = pd.read_sql(query3, conn)
resultat3

,heure,nombre_commandes
0,12,3220
1,13,2753
2,14,2457
3,11,2396
4,10,2361
5,15,2336
6,09,1484
7,16,1335
8,17,667
9,08,566


In [17]:
# Clients fidèles versus clients occasionnels 
# Combien de clients reviennent acheter plusieur fois ?

query4 = """
WITH commandes_clients AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS nombre_commandes
    FROM sales
    WHERE CustomerID IS NOT NULL
      AND Quantity > 0
      AND UnitPrice > 0
      AND InvoiceNo NOT LIKE 'C%'
    GROUP BY CustomerID
)
SELECT
    CASE
        WHEN nombre_commandes >= 2 THEN 'Client fidele'
        ELSE 'Client occasionnel'
    END AS segment_client,
    COUNT(*) AS nombre_clients
FROM commandes_clients
GROUP BY segment_client;
"""

resultat4 = pd.read_sql(query4, conn)
resultat4

,segment_client,nombre_clients
0,Client fidele,2845
1,Client occasionnel,1493


In [18]:
# Panier moyen par commande 

query5 = """
WITH commandes AS (
    SELECT
        InvoiceNo,
        SUM(Quantity * UnitPrice) AS montant_commande
    FROM sales
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
    GROUP BY InvoiceNo
)
SELECT
    COUNT(*) AS nombre_commandes,
    ROUND(AVG(montant_commande), 2) AS panier_moyen
FROM commandes;
"""

resultat5 = pd.read_sql(query5, conn)
resultat5


,nombre_commandes,panier_moyen
0,19960,533.17


In [ ]:
conn.close()
print("connexion SQLite fermée.")

In [21]:
from pathlib import Path

print("Dossier courant :", Path.cwd())
print("Base de données :", Path("../analysis.db").resolve())
print("Base créée :", Path("../analysis.db").exists())

Dossier courant : C:\Users\hp\Documents\Projet_Python\notebooks
Base de données : C:\Users\hp\Documents\Projet_Python\analysis.db
Base créée : True
